In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *


spark.conf.set(
    "spark.sql.session.timeZone",
    "UTC"
)


catalog = "crypto_lakehouse"
bronze_schema = "bronze"
silver_schema = "silver"


spark.sql(
    f"""
    CREATE SCHEMA IF NOT EXISTS
    {catalog}.{silver_schema}
    """
)

DataFrame[]

####CoinDCX balances

In [0]:
balances_bronze_df = spark.table(
    f"{catalog}.{bronze_schema}.coindcx_balances"
)


print(
    "CoinDCX Bronze balance rows:",
    balances_bronze_df.count()
)

balances_bronze_df.printSchema()

display(balances_bronze_df)

CoinDCX Bronze balance rows: 4
root
 |-- balance: decimal(38,18) (nullable = true)
 |-- currency: string (nullable = true)
 |-- locked_balance: decimal(38,18) (nullable = true)
 |-- source_file_name: string (nullable = true)
 |-- _source_file: string (nullable = true)
 |-- exchange: string (nullable = true)
 |-- source_type: string (nullable = true)
 |-- ingested_at: timestamp (nullable = true)



balance,currency,locked_balance,source_file_name,_source_file,exchange,source_type,ingested_at
0.121948825900600000,SOL,0E-18,balances_20260925T181928_277012Z_c83f7988.json,s3://l1l2bucket/data/raw/coindcx/balances/balances_20260925T181928_277012Z_c83f7988.json,CoinDCX,Balance,2026-09-25T18:26:39.618Z
0E-18,BTC,0E-18,balances_20260925T181928_277012Z_c83f7988.json,s3://l1l2bucket/data/raw/coindcx/balances/balances_20260925T181928_277012Z_c83f7988.json,CoinDCX,Balance,2026-09-25T18:26:39.618Z
0.034810000000000000,INR,0E-18,balances_20260925T181928_277012Z_c83f7988.json,s3://l1l2bucket/data/raw/coindcx/balances/balances_20260925T181928_277012Z_c83f7988.json,CoinDCX,Balance,2026-09-25T18:26:39.618Z
0E-18,USDT,0E-18,balances_20260925T181928_277012Z_c83f7988.json,s3://l1l2bucket/data/raw/coindcx/balances/balances_20260925T181928_277012Z_c83f7988.json,CoinDCX,Balance,2026-09-25T18:26:39.618Z


In [0]:
coindcx_balances_silver_df = (
    balances_bronze_df
    .select(
        col("source_file_name")
        .alias("snapshot_id"),

        lit(None)
        .cast("string")
        .alias("account_id"),

        upper(
            col("currency")
        ).alias("asset"),

        col("balance")
        .cast("decimal(38,18)")
        .alias("free_balance"),

        col("locked_balance")
        .cast("decimal(38,18)")
        .alias("locked_balance"),

        col("exchange"),

        col("_source_file"),

        col("ingested_at")
        .alias("bronze_ingested_at")
    )
    .withColumn(
        "total_balance",
        col("free_balance")
        + col("locked_balance")
    )
    .withColumn(
        "silver_processed_at",
        current_timestamp()
    )
)


print(
    "CoinDCX Silver balance rows:",
    coindcx_balances_silver_df.count()
)

display(
    coindcx_balances_silver_df.orderBy(
        col("total_balance").desc()
    )
)

CoinDCX Silver balance rows: 4


snapshot_id,account_id,asset,free_balance,locked_balance,exchange,_source_file,bronze_ingested_at,total_balance,silver_processed_at
balances_20260925T181928_277012Z_c83f7988.json,null,SOL,0.121948825900600000,0E-18,CoinDCX,s3://l1l2bucket/data/raw/coindcx/balances/balances_20260925T181928_277012Z_c83f7988.json,2026-09-25T18:26:39.618Z,0.12194882590060000,2026-10-01T15:48:09.027Z
balances_20260925T181928_277012Z_c83f7988.json,null,INR,0.034810000000000000,0E-18,CoinDCX,s3://l1l2bucket/data/raw/coindcx/balances/balances_20260925T181928_277012Z_c83f7988.json,2026-09-25T18:26:39.618Z,0.03481000000000000,2026-10-01T15:48:09.027Z
balances_20260925T181928_277012Z_c83f7988.json,null,BTC,0E-18,0E-18,CoinDCX,s3://l1l2bucket/data/raw/coindcx/balances/balances_20260925T181928_277012Z_c83f7988.json,2026-09-25T18:26:39.618Z,0E-17,2026-10-01T15:48:09.027Z
balances_20260925T181928_277012Z_c83f7988.json,null,USDT,0E-18,0E-18,CoinDCX,s3://l1l2bucket/data/raw/coindcx/balances/balances_20260925T181928_277012Z_c83f7988.json,2026-09-25T18:26:39.618Z,0E-17,2026-10-01T15:48:09.027Z


In [0]:
coindcx_balances_final_df = (
    coindcx_balances_silver_df
    .dropDuplicates([
        "snapshot_id",
        "account_id",
        "asset"
    ])
)


(
    coindcx_balances_final_df
    .write
    .format("delta")
    .mode("overwrite")
    .option(
        "overwriteSchema",
        True
    )
    .saveAsTable(
        f"{catalog}.{silver_schema}.coindcx_account_balances"
    )
)


print(
    "Saved CoinDCX balance rows:",
    spark.table(
        f"{catalog}.{silver_schema}.coindcx_account_balances"
    ).count()
)

Saved CoinDCX balance rows: 4


####CoinDCX trades

In [0]:
instant_orders_bronze_df = spark.table(
    f"{catalog}.{bronze_schema}.coindcx_instant_orders"
)


print(
    "CoinDCX Bronze Instant order rows:",
    instant_orders_bronze_df.count()
)

instant_orders_bronze_df.printSchema()

display(instant_orders_bronze_df)

CoinDCX Bronze Instant order rows: 1
root
 |-- transactionId: string (nullable = true)
 |-- tradeSource: string (nullable = true)
 |-- asset: string (nullable = true)
 |-- quoteAsset: string (nullable = true)
 |-- side: string (nullable = true)
 |-- orderType: string (nullable = true)
 |-- status: string (nullable = true)
 |-- quantity: string (nullable = true)
 |-- averagePriceInr: string (nullable = true)
 |-- grossAmountInr: string (nullable = true)
 |-- feeInr: string (nullable = true)
 |-- netAmountInr: string (nullable = true)
 |-- tdsInr: string (nullable = true)
 |-- transactionTime: string (nullable = true)
 |-- source_file_name: string (nullable = true)
 |-- _source_file: string (nullable = true)
 |-- exchange: string (nullable = true)
 |-- source_type: string (nullable = true)
 |-- ingested_at: timestamp (nullable = true)



transactionId,tradeSource,asset,quoteAsset,side,orderType,status,quantity,averagePriceInr,grossAmountInr,feeInr,netAmountInr,tdsInr,transactionTime,source_file_name,_source_file,exchange,source_type,ingested_at
292a7605-4955-4d01-b296-41d32c14920e,Instant,SOL,INR,BUY,INSTANT,COMPLETED,0.1219488259006,8151.78,994.10,5.86519,999.96519,null,2026-03-14T21:20:21+05:30,instant_orders_20260925T181928_600854Z_a4900abc.json,s3://l1l2bucket/data/raw/coindcx/instant_orders/instant_orders_20260925T181928_600854Z_a4900abc.json,CoinDCX,Instant Order,2026-09-25T18:26:51.001Z


In [0]:
coindcx_trades_silver_df = (
    instant_orders_bronze_df
    .filter(
        upper(col("status"))
        == "COMPLETED"
    )
    .select(
        col("transactionId")
        .alias("transaction_id"),

        col("tradeSource")
        .alias("trade_source"),

        concat_ws(
            "/",
            upper(col("asset")),
            upper(col("quoteAsset"))
        ).alias("symbol"),

        upper(col("side"))
        .alias("side"),

        when(
            upper(col("side")) == "BUY",
            upper(col("quoteAsset"))
        )
        .otherwise(
            upper(col("asset"))
        )
        .alias("asset_spent"),

        when(
            upper(col("side")) == "BUY",
            col("grossAmountInr")
        )
        .otherwise(
            col("quantity")
        )
        .cast("decimal(38,18)")
        .alias("amount_spent"),

        when(
            upper(col("side")) == "BUY",
            upper(col("asset"))
        )
        .otherwise(
            upper(col("quoteAsset"))
        )
        .alias("asset_received"),

        when(
            upper(col("side")) == "BUY",
            col("quantity")
        )
        .otherwise(
            col("grossAmountInr")
        )
        .cast("decimal(38,18)")
        .alias("amount_received"),

        col("feeInr")
        .cast("decimal(38,18)")
        .alias("fee_amount"),

        lit("INR")
        .alias("fee_asset"),

        to_timestamp(
            col("transactionTime")
        ).alias("trade_time"),

        col("exchange"),
        col("_source_file"),

        col("ingested_at")
        .alias("bronze_ingested_at")
    )
)


print(
    "CoinDCX Silver trade rows:",
    coindcx_trades_silver_df.count()
)

display(coindcx_trades_silver_df)

CoinDCX Silver trade rows: 1


transaction_id,trade_source,symbol,side,asset_spent,amount_spent,asset_received,amount_received,fee_amount,fee_asset,trade_time,exchange,_source_file,bronze_ingested_at
292a7605-4955-4d01-b296-41d32c14920e,Instant,SOL/INR,BUY,INR,994.100000000000000000,SOL,0.121948825900600000,5.865190000000000000,INR,2026-03-14T15:50:21.000Z,CoinDCX,s3://l1l2bucket/data/raw/coindcx/instant_orders/instant_orders_20260925T181928_600854Z_a4900abc.json,2026-09-25T18:26:51.001Z


In [0]:
coindcx_trades_final_df = (
    coindcx_trades_silver_df
    .filter(
        col("transaction_id").isNotNull()
        & col("asset_spent").isNotNull()
        & col("asset_received").isNotNull()
        & (col("amount_spent") > 0)
        & (col("amount_received") > 0)
    )
    .withColumn(
        "trade_key",
        concat_ws(
            "|",
            col("exchange"),
            col("trade_source"),
            col("symbol"),
            col("transaction_id")
        )
    )
    .withColumn(
        "trade_date",
        to_date(
            col("trade_time")
        )
    )
    .withColumn(
        "silver_processed_at",
        current_timestamp()
    )
    .dropDuplicates([
        "trade_key"
    ])
)


print(
    "Final CoinDCX trade rows:",
    coindcx_trades_final_df.count()
)

display(coindcx_trades_final_df)

Final CoinDCX trade rows: 1


transaction_id,trade_source,symbol,side,asset_spent,amount_spent,asset_received,amount_received,fee_amount,fee_asset,trade_time,exchange,_source_file,bronze_ingested_at,trade_key,trade_date,silver_processed_at
292a7605-4955-4d01-b296-41d32c14920e,Instant,SOL/INR,BUY,INR,994.100000000000000000,SOL,0.121948825900600000,5.865190000000000000,INR,2026-03-14T15:50:21.000Z,CoinDCX,s3://l1l2bucket/data/raw/coindcx/instant_orders/instant_orders_20260925T181928_600854Z_a4900abc.json,2026-09-25T18:26:51.001Z,CoinDCX|Instant|SOL/INR|292a7605-4955-4d01-b296-41d32c14920e,2026-03-14,2026-10-01T15:48:19.093Z


In [0]:
(
    coindcx_trades_final_df
    .write
    .format("delta")
    .mode("overwrite")
    .option(
        "overwriteSchema",
        True
    )
    .saveAsTable(
        f"{catalog}.{silver_schema}.coindcx_trades"
    )
)


print(
    "Saved CoinDCX trades:",
    spark.table(
        f"{catalog}.{silver_schema}.coindcx_trades"
    ).count()
)

Saved CoinDCX trades: 1


####CoinDCX fiat deposits

In [0]:
fiat_deposits_bronze_df = spark.table(
    f"{catalog}.{bronze_schema}.coindcx_fiat_deposits"
)


coindcx_fiat_deposits_silver_df = (
    fiat_deposits_bronze_df
    .select(
        col("transactionId")
        .alias("deposit_id"),

        upper(col("asset"))
        .alias("asset"),

        col("amount")
        .cast("decimal(38,18)")
        .alias("amount"),

        lit(0)
        .cast("decimal(38,18)")
        .alias("fee_amount"),

        upper(col("status"))
        .alias("status"),

        to_timestamp(
            col("createdAt")
        ).alias("deposit_time"),

        col("exchange"),
        col("_source_file"),

        col("ingested_at")
        .alias("bronze_ingested_at")
    )
    .withColumn(
        "silver_processed_at",
        current_timestamp()
    )
)


print(
    "CoinDCX Silver fiat deposit rows:",
    coindcx_fiat_deposits_silver_df.count()
)

display(coindcx_fiat_deposits_silver_df)

coindcx_fiat_deposits_silver_df.printSchema()

CoinDCX Silver fiat deposit rows: 1


deposit_id,asset,amount,fee_amount,status,deposit_time,exchange,_source_file,bronze_ingested_at,silver_processed_at
422144727195,INR,1000.000000000000000000,0E-18,COMPLETED,2026-03-14T15:36:00.000Z,CoinDCX,s3://l1l2bucket/data/raw/coindcx/fiat_deposits/fiat_deposits_20260925T181928_745720Z_6855d341.json,2026-09-25T18:27:01.891Z,2026-10-01T15:48:25.415Z


root
 |-- deposit_id: string (nullable = true)
 |-- asset: string (nullable = true)
 |-- amount: decimal(38,18) (nullable = true)
 |-- fee_amount: decimal(38,18) (nullable = false)
 |-- status: string (nullable = true)
 |-- deposit_time: timestamp (nullable = true)
 |-- exchange: string (nullable = true)
 |-- _source_file: string (nullable = true)
 |-- bronze_ingested_at: timestamp (nullable = true)
 |-- silver_processed_at: timestamp (nullable = false)

Error in callback <bound method UserNamespaceCommandHook.post_run_cell of <dbruntime.DatasetInfo.UserNamespaceCommandHook object at 0xffe511ea0410>> (for post_run_cell), with arguments args (<ExecutionResult object at ffe4fc39bb90, execution_count=10 error_before_exec=None error_in_exec=None info=<ExecutionInfo object at ffe4fc3986b0, raw_cell="fiat_deposits_bronze_df = spark.table(
    f"{cata.." store_history=True silent=False shell_futures=True cell_id=6579444567747949> result=None>,),kwargs {}:


In [0]:
coindcx_fiat_deposits_final_df = (
    coindcx_fiat_deposits_silver_df
    .filter(
        col("deposit_id").isNotNull()
        & (col("amount") > 0)
    )
    .dropDuplicates([
        "deposit_id"
    ])
)


(
    coindcx_fiat_deposits_final_df
    .write
    .format("delta")
    .mode("overwrite")
    .option(
        "overwriteSchema",
        True
    )
    .saveAsTable(
        f"{catalog}.{silver_schema}.coindcx_fiat_deposits"
    )
)


print(
    "Saved CoinDCX fiat deposits:",
    spark.table(
        f"{catalog}.{silver_schema}.coindcx_fiat_deposits"
    ).count()
)

####CoinDCX fiat withdrawal 

In [0]:
fiat_withdrawals_bronze_df = spark.table(
    f"{catalog}.{bronze_schema}.coindcx_fiat_withdrawals"
)

coindcx_fiat_withdrawals_silver_df = (
    fiat_withdrawals_bronze_df
    .select(
        col("transactionId").alias("withdrawal_id"),

        upper(col("asset")).alias("asset"),

        col("amount")
        .cast("decimal(38,18)")
        .alias("amount"),

        lit(0)
        .cast("decimal(38,18)")
        .alias("fee_amount"),

        upper(col("status")).alias("status"),

        to_timestamp(
            col("createdAt")
        ).alias("withdrawal_time"),

        col("exchange"),
        col("_source_file"),

        col("ingested_at")
        .alias("bronze_ingested_at")
    )
    .withColumn(
        "silver_processed_at",
        current_timestamp()
    )
    .dropDuplicates([
        "withdrawal_id"
    ])
)

print(
    "Clean fiat withdrawal rows:",
    coindcx_fiat_withdrawals_silver_df.count()
)

coindcx_fiat_withdrawals_silver_df.printSchema()

(
    coindcx_fiat_withdrawals_silver_df
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", True)
    .saveAsTable(
        f"{catalog}.{silver_schema}.coindcx_fiat_withdrawals"
    )
)

print(
    "Saved fiat withdrawal rows:",
    spark.table(
        f"{catalog}.{silver_schema}.coindcx_fiat_withdrawals"
    ).count()
)

####CoinDCX crypto deposits

In [0]:
crypto_deposits_bronze_df = spark.table(
    f"{catalog}.{bronze_schema}.coindcx_crypto_deposits"
)

coindcx_crypto_deposits_silver_df = (
    crypto_deposits_bronze_df
    .select(
        col("transactionId").alias("deposit_id"),

        col("transactionHash")
        .alias("transaction_hash"),

        upper(col("asset")).alias("asset"),

        col("amount")
        .cast("decimal(38,18)")
        .alias("amount"),

        col("fee")
        .cast("decimal(38,18)")
        .alias("fee_amount"),

        upper(col("network")).alias("network"),

        upper(col("status")).alias("status"),

        col("fromAddress")
        .alias("source_address"),

        col("toAddress")
        .alias("deposit_address"),

        to_timestamp(
            col("createdAt")
        ).alias("deposit_time"),

        col("exchange"),
        col("_source_file"),

        col("ingested_at")
        .alias("bronze_ingested_at")
    )
    .withColumn(
        "silver_processed_at",
        current_timestamp()
    )
    .dropDuplicates([
        "deposit_id"
    ])
)

print(
    "Clean crypto deposit rows:",
    coindcx_crypto_deposits_silver_df.count()
)

coindcx_crypto_deposits_silver_df.printSchema()

(
    coindcx_crypto_deposits_silver_df
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", True)
    .saveAsTable(
        f"{catalog}.{silver_schema}.coindcx_crypto_deposits"
    )
)

print(
    "Saved crypto deposit rows:",
    spark.table(
        f"{catalog}.{silver_schema}.coindcx_crypto_deposits"
    ).count()
)

####CoinDCX crypto withdrawals

In [0]:
crypto_withdrawals_bronze_df = spark.table(
    f"{catalog}.{bronze_schema}.coindcx_crypto_withdrawals"
)

coindcx_crypto_withdrawals_silver_df = (
    crypto_withdrawals_bronze_df
    .select(
        col("transactionId")
        .alias("withdrawal_id"),

        col("transactionHash")
        .alias("transaction_hash"),

        upper(col("asset"))
        .alias("asset"),

        col("amount")
        .cast("decimal(38,18)")
        .alias("amount"),

        col("fee")
        .cast("decimal(38,18)")
        .alias("transaction_fee"),

        upper(col("network"))
        .alias("network"),

        upper(col("status"))
        .alias("status"),

        col("fromAddress")
        .alias("from_address"),

        col("toAddress")
        .alias("withdrawal_address"),

        to_timestamp(
            col("createdAt")
        ).alias("withdrawal_time"),

        col("exchange"),
        col("_source_file"),

        col("ingested_at")
        .alias("bronze_ingested_at")
    )
    .withColumn(
        "silver_processed_at",
        current_timestamp()
    )
    .dropDuplicates([
        "withdrawal_id"
    ])
)

print(
    "Clean crypto withdrawal rows:",
    coindcx_crypto_withdrawals_silver_df.count()
)

coindcx_crypto_withdrawals_silver_df.printSchema()

(
    coindcx_crypto_withdrawals_silver_df
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", True)
    .saveAsTable(
        f"{catalog}.{silver_schema}.coindcx_crypto_withdrawals"
    )
)

print(
    "Saved crypto withdrawal rows:",
    spark.table(
        f"{catalog}.{silver_schema}.coindcx_crypto_withdrawals"
    ).count()
)